Варіант 5 — Дніпро.

Позначимо:

x1 — кількість одиниць виробу А, які виготовляються
за тиждень;

x2 — кількість одиниць виробу Б, які виготовляються
за тиждень.

Прибуток від одного виробу А становить 44 грн,
а від одного виробу Б — 48 грн.

Тому цільова функція:

максимізувати

Z = 44x1 + 48x2.

Обмеження за сировиною:

4x1 + 3x2 <= 195.

Обмеження за робочим часом обладнання:

2x1 + 4x2 <= 160.

Обмеження за електроенергією:

x1 + x2 <= 70.

Також кількість продукції не може бути від'ємною:

x1 >= 0,
x2 >= 0.

Отже, задача лінійного програмування:

max Z = 44x1 + 48x2

при умовах:

4x1 + 3x2 <= 195
2x1 + 4x2 <= 160
x1 + x2 <= 70
x1 >= 0
x2 >= 0.

In [1]:
from scipy.optimize import linprog
import numpy as np

# Для максимізації 44*x1 + 48*x2
# мінімізуємо -(44*x1 + 48*x2)

c = [-44, -48]

# Обмеження
A_ub = [
    [4, 3],   # сировина
    [2, 4],   # час
    [1, 1]    # електроенергія
]

b_ub = [
    195,  # кг сировини
    160,  # машино-годин
    70    # кВт·год
]

bounds = [
    (0, None),
    (0, None)
]

res = linprog(
    c,
    A_ub=A_ub,
    b_ub=b_ub,
    bounds=bounds,
    method="highs"
)

print("res.x =", res.x)
print("res.fun =", res.fun)
print("Максимальний прибуток =", -res.fun)
print("res.status =", res.status)
print("res.message =", res.message)

res.x = [30. 25.]
res.fun = -2520.0
Максимальний прибуток = 2520.0
res.status = 0
res.message = Optimization terminated successfully. (HiGHS Status 7: Optimal)


Функція linprog успішно знайшла оптимальний розв'язок.

res.status = 0, що означає успішне завершення
оптимізації та знаходження оптимального розв'язку.

Оскільки linprog мінімізує функцію, у коді
використовувались коефіцієнти [-44, -48].

Тому res.fun = -2520, а справжній максимальний
прибуток становить:

-res.fun = 2520 грн на тиждень.

In [2]:
x1, x2 = res.x
max_profit = -res.fun

print(f"Виріб А: {x1:.0f} од.")
print(f"Виріб Б: {x2:.0f} од.")
print(f"Максимальний прибуток: {max_profit:.0f} грн")

Виріб А: 30 од.
Виріб Б: 25 од.
Максимальний прибуток: 2520 грн


Оптимальний план виробництва:

виріб А — 30 одиниць на тиждень;
виріб Б — 25 одиниць на тиждень.

При цьому максимальний тижневий прибуток становить:

44 × 30 + 48 × 25 =
1320 + 1200 =
2520 грн.

У цьому варіанті оптимальні значення x1 і x2 вже
є цілими числами, тому додаткове округлення
не потрібне.

Якби linprog повернув дробові значення, просте
округлення не гарантувало б найкращого цілочислового
розв'язку.

Після округлення план може порушити ресурсні
обмеження або дати менший прибуток, ніж інша
допустима комбінація цілих значень.

Для задач, де кількість продукції обов'язково має
бути цілою, коректніше використовувати методи
цілочислового програмування.

In [3]:
print("Запаси ресурсів (slack):")
print(res.slack)

Запаси ресурсів (slack):
[ 0.  0. 15.]


In [4]:
raw_used = 4*x1 + 3*x2
time_used = 2*x1 + 4*x2
energy_used = x1 + x2

print(
    f"Сировина: використано {raw_used:.0f} "
    f"із 195 кг"
)

print(
    f"Час: використано {time_used:.0f} "
    f"із 160 год"
)

print(
    f"Електроенергія: використано {energy_used:.0f} "
    f"із 70 кВт·год"
)

Сировина: використано 195 із 195 кг
Час: використано 160 із 160 год
Електроенергія: використано 55 із 70 кВт·год


Для оптимального плану отримано:

сировина:
4 × 30 + 3 × 25 = 195 кг із 195 кг;

робочий час:
2 × 30 + 4 × 25 = 160 год із 160 год;

електроенергія:
30 + 25 = 55 кВт·год із 70 кВт·год.

Значення res.slack:

[0, 0, 15].

Отже, обмеження за сировиною та робочим часом
є активними, оскільки їх slack = 0 і ці ресурси
використовуються повністю.

Обмеження за електроенергією є неактивним.
Залишається запас 15 кВт·год.

Таким чином, сировина і робочий час є вузькими
місцями виробництва.

Збільшення доступної сировини або робочого часу
потенційно може дозволити збільшити прибуток.

Саме по собі збільшення ліміту електроенергії
не повинно покращити оптимальний результат,
оскільки вже зараз частина цього ресурсу
залишається невикористаною.

In [5]:
b_active = [
    195 * 1.15,  # сировина +15%
    160,
    70
]

res_active = linprog(
    c,
    A_ub=A_ub,
    b_ub=b_active,
    bounds=bounds,
    method="highs"
)

print("Новий план:", res_active.x)
print("Новий прибуток:", -res_active.fun)
print("Новий slack:", res_active.slack)

Новий план: [41.7  19.15]
Новий прибуток: 2754.0
Новий slack: [0.   0.   9.15]


In [6]:
print("СТАРИЙ РЕЗУЛЬТАТ")
print("A =", res.x[0])
print("Б =", res.x[1])
print("Прибуток =", -res.fun)

print("\nПІСЛЯ +15% СИРОВИНИ")
print("A =", res_active.x[0])
print("Б =", res_active.x[1])
print("Прибуток =", -res_active.fun)

print(
    "\nЗбільшення прибутку =",
    -res_active.fun - (-res.fun)
)

СТАРИЙ РЕЗУЛЬТАТ
A = 30.000000000000004
Б = 24.999999999999996
Прибуток = 2520.0

ПІСЛЯ +15% СИРОВИНИ
A = 41.69999999999999
Б = 19.150000000000006
Прибуток = 2754.0

Збільшення прибутку = 234.0


In [7]:
increase = (
    (-res_active.fun - (-res.fun))
    / (-res.fun)
    * 100
)

print(f"Зростання прибутку: {increase:.2f}%")

Зростання прибутку: 9.29%


Сировина була активним обмеженням, тому її ліміт
збільшено на 15%:

195 × 1.15 = 224.25 кг.

Після повторної оптимізації отримано:

виріб А ≈ 41.70 од.;
виріб Б ≈ 19.15 од.;

максимальний прибуток = 2754 грн.

Початковий прибуток становив 2520 грн.

Отже, збільшення ліміту сировини на 15% підвищило
максимальний прибуток на:

2754 - 2520 = 234 грн,

або приблизно на 9.29%.

Структура виробництва також змінилася.

Спочатку:

А = 30;
Б = 25.

Після збільшення сировини:

А ≈ 41.70;
Б ≈ 19.15.

Тобто підприємству стало вигідно виробляти більше
виробу А і менше виробу Б.

Цей результат підтверджує, що сировина справді
була одним із вузьких місць виробництва.

In [8]:
b_inactive = [
    195,
    160,
    70 * 1.5   # електроенергія +50%
]

res_inactive = linprog(
    c,
    A_ub=A_ub,
    b_ub=b_inactive,
    bounds=bounds,
    method="highs"
)

print("Новий план:", res_inactive.x)
print("Новий прибуток:", -res_inactive.fun)
print("Новий slack:", res_inactive.slack)

Новий план: [30. 25.]
Новий прибуток: 2520.0
Новий slack: [ 0.  0. 50.]


Електроенергія була неактивним обмеженням.

Її початковий ліміт становив 70 кВт·год, а фактично
в оптимальному плані використовувалося лише
55 кВт·год.

Збільшимо ліміт на 50%:

70 × 1.5 = 105 кВт·год.

Після повторної оптимізації отримано той самий план:

виріб А = 30 од.;
виріб Б = 25 од.

Максимальний прибуток також не змінився:

2520 грн.

Причина полягає в тому, що електроенергія не була
вузьким місцем виробництва.

До збільшення ліміту вже існував запас 15 кВт·год,
тому додаткова електроенергія не дозволяє виробити
більше продукції.

Виробництво все одно обмежується доступною сировиною
та робочим часом.

Отже, збільшення неактивного ресурсу не покращило
оптимальний результат.

1. Чому scipy.optimize.linprog мінімізує і що
потрібно зробити для максимізації?

Функція scipy.optimize.linprog розв'язує задачу
у формі мінімізації цільової функції.

У нашій задачі потрібно максимізувати:

Z = 44x1 + 48x2.

Тому замість цього передаємо функції:

-Z = -44x1 - 48x2.

У коді:

c = [-44, -48].

Після оптимізації res.fun є мінімальним значенням
функції -Z, тому справжній максимальний прибуток
визначається як:

-res.fun.

У нашому випадку:

res.fun = -2520,

тому максимальний прибуток:

Zmax = 2520 грн.


2. Чому обмеження виду >= потрібно множити на -1?

Параметри A_ub і b_ub функції linprog представляють
обмеження у стандартній формі:

A_ub @ x <= b_ub.

Тому якщо початкове обмеження записано через >=,
його потрібно перетворити на <=.

Наприклад:

2x1 + 3x2 >= 10.

Помножимо обидві частини на -1:

-2x1 - 3x2 <= -10.

Після цього обмеження можна коректно передати
через A_ub і b_ub.

У нашому варіанті всі три ресурсні обмеження
відразу записані через <=, тому змінювати
їхній знак не потрібно.


3. Чим відрізняються активні та неактивні обмеження?

Активне обмеження в оптимальній точці виконується
точно на межі, тобто ресурс використовується повністю.

Для нього slack = 0.

У нашій задачі активними є:

сировина — 195 із 195 кг;
робочий час — 160 із 160 год.

Ці ресурси є вузькими місцями виробництва.

Неактивне обмеження має запас, тобто slack > 0.

У нашій задачі електроенергії використано лише
55 із 70 кВт·год, тому slack = 15.

Через це збільшення саме активного ресурсу може
дозволити збільшити прибуток, тоді як збільшення
неактивного ресурсу саме по собі результат
не покращить.


4. Чи узгоджуються результати Завдання 5 з поняттям
активного обмеження?

Так.

Сировина була активним обмеженням із slack = 0.

Після збільшення її ліміту на 15% максимальний
прибуток зріс:

з 2520 грн до 2754 грн.

Отже, додаткова сировина дала можливість змінити
структуру виробництва і збільшити прибуток.

Електроенергія була неактивним обмеженням
із початковим запасом 15 кВт·год.

Після збільшення її ліміту на 50% оптимальний
план залишився:

А = 30;
Б = 25,

а прибуток залишився 2520 грн.

Отже, результати повністю узгоджуються з поняттям
вузького місця: розширення активного ресурсу
може покращити результат, а збільшення ресурсу,
якого вже є в надлишку, результат не змінює.